# Tutorial: Phase 1 ViewState + Axis Selectors

Audience:
- Engineers validating state/session/view behavior for Lucida phase 1.

Prerequisites:
- Run `uv sync` in this repository.

Learning goals:
- Open a dataset in a session.
- Create and mutate a 2D ViewState through axis selector updates.
- Verify selector normalization, hash updates, and version increments.


## Step 1 - Imports and OME-Zarr fixture helper


In [1]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path

import numpy as np
import zarr
from fastapi.testclient import TestClient

from lucida.client import LucidaClient
from lucida.server.app import create_app
from lucida.service.dataset_service import DatasetService


def build_demo_omezarr(dataset_path: Path) -> str:
    dataset_path.mkdir(parents=True, exist_ok=True)
    root = zarr.open_group(store=str(dataset_path), mode='w')

    shape_0 = (1, 2, 4, 8, 10)
    shape_1 = (1, 2, 2, 4, 5)
    root.create_array(
        '0',
        data=np.arange(np.prod(shape_0), dtype=np.uint16).reshape(shape_0),
        chunks=(1, 1, 2, 4, 5),
        overwrite=True,
    )
    root.create_array(
        '1',
        data=np.arange(np.prod(shape_1), dtype=np.uint16).reshape(shape_1),
        chunks=(1, 1, 1, 2, 3),
        overwrite=True,
    )

    root.attrs['multiscales'] = [
        {
            'name': 'primary',
            'axes': [
                {'name': 't', 'type': 't'},
                {'name': 'c', 'type': 'c'},
                {'name': 'z', 'type': 'z', 'unit': 'micron'},
                {'name': 'y', 'type': 'y', 'unit': 'micron'},
                {'name': 'x', 'type': 'x', 'unit': 'micron'},
            ],
            'datasets': [
                {
                    'path': '0',
                    'coordinateTransformations': [
                        {'type': 'scale', 'scale': [1, 1, 1, 1, 1]},
                        {'type': 'translation', 'translation': [0, 0, 0, 0, 0]},
                    ],
                },
                {
                    'path': '1',
                    'coordinateTransformations': [
                        {'type': 'scale', 'scale': [1, 1, 2, 2, 2]}
                    ],
                },
            ],
        }
    ]
    root.attrs['omero'] = {
        'channels': [
            {'index': 0, 'label': 'DNA', 'color': 'FF0000', 'window': {'start': 10, 'end': 400}},
            {'index': 1, 'label': 'RNA', 'color': '00FF00', 'window': {'start': 20, 'end': 200}},
        ]
    }

    return str(dataset_path)


## Step 2 - Build dataset and initialize an in-process API


In [2]:
tmp_dir = Path(tempfile.mkdtemp(prefix='lucida-viewstate-'))
dataset_uri = build_demo_omezarr(tmp_dir / 'viewstate-demo.zarr')

service = DatasetService()
test_client = TestClient(create_app(dataset_service=service))
lucida = LucidaClient(client=test_client)

dataset_uri


'/var/folders/hs/qw7ws1q52153c4c639t_p3600000gn/T/lucida-viewstate-0cml_ig0/viewstate-demo.zarr'

## Step 3 - Create session, open dataset, and create a view


In [3]:
session = lucida.create_session()
opened = lucida.open_dataset(uri=dataset_uri, session_id=session.session_id)
created = lucida.create_view(
    dataset_id=opened.dataset_summary.dataset_id,
    session_id=session.session_id,
    mode='2d',
)

print('session_id:', session.session_id)
print('dataset_id:', opened.dataset_summary.dataset_id)
print('view_id:', created.view_state.view_id)
print('initial state_version:', created.view_state.state_version)
print('initial state_hash:', created.view_state.state_hash)


session_id: session_6bbf8a8e5a9f4e79
dataset_id: ds_694f0c2aaadaf5e2
view_id: view_0a3bbda18b3f46d7
initial state_version: 0
initial state_hash: 20b4ca1bd345f1ef23d05b35844fc82d0f96c6aada221ec51a4ef00e1c200e5f


## Step 4 - Apply selector helpers and validate invariants


In [4]:
updated_index = lucida.set_dim(
    view_id=created.view_state.view_id,
    axis='z',
    index=2,
    session_id=session.session_id,
)
updated_range = lucida.set_axis_range(
    view_id=created.view_state.view_id,
    axis='z',
    start=1,
    end_exclusive=4,
    session_id=session.session_id,
)
updated_set = lucida.set_axis_set(
    view_id=created.view_state.view_id,
    axis='z',
    indices=[0, 2, 2, 3],
    session_id=session.session_id,
)

z_index_selector = next(s for s in updated_index.selectors_applied if s.axis == 'z')
z_range_selector = next(s for s in updated_range.selectors_applied if s.axis == 'z')
z_set_selector = next(s for s in updated_set.selectors_applied if s.axis == 'z')

assert z_index_selector.index == 2
assert z_range_selector.start == 1 and z_range_selector.end_exclusive == 4
assert z_set_selector.indices == [0, 2, 3]

assert created.view_state.state_version == 0
assert updated_index.view_state.state_version == 1
assert updated_range.view_state.state_version == 2
assert updated_set.view_state.state_version == 3

assert created.view_state.state_hash != updated_index.view_state.state_hash
assert updated_index.view_state.state_hash != updated_range.view_state.state_hash
assert updated_range.view_state.state_hash != updated_set.view_state.state_hash

print('Selector and version/hash assertions passed.')


Selector and version/hash assertions passed.


## Step 5 - Retrieve view and inspect payload


In [5]:
fetched = lucida.get_view(view_id=created.view_state.view_id, session_id=session.session_id)
assert fetched.view_state.view_id == created.view_state.view_id
assert fetched.view_state.session_id == session.session_id
assert fetched.view_state.state_version == 3

preview = fetched.view_state.model_dump(mode='json')
print(json.dumps({
    'view_id': preview['view_id'],
    'session_id': preview['session_id'],
    'state_version': preview['state_version'],
    'state_hash': preview['state_hash'],
    'selectors': preview['selectors'],
}, indent=2))


{
  "view_id": "view_0a3bbda18b3f46d7",
  "session_id": "session_6bbf8a8e5a9f4e79",
  "state_version": 3,
  "state_hash": "c2771a4f789471cee1678648d1b792b801f69184be188e87988911585ed92c5a",
  "selectors": [
    {
      "axis": "t",
      "kind": "index",
      "index": 0,
      "start": null,
      "end_exclusive": null,
      "indices": null,
      "clamp": true
    },
    {
      "axis": "c",
      "kind": "index",
      "index": 0,
      "start": null,
      "end_exclusive": null,
      "indices": null,
      "clamp": true
    },
    {
      "axis": "z",
      "kind": "set",
      "index": null,
      "start": null,
      "end_exclusive": null,
      "indices": [
        0,
        2,
        3
      ],
      "clamp": true
    }
  ]
}


## Expected output checks

After running all cells, verify:
- Non-empty `session_id`, `dataset_id`, and `view_id` are printed.
- `Selector and version/hash assertions passed.` is printed.
- Final `state_version` equals `3`.
- Final selectors include a normalized `z` set selector `[0, 2, 3]`.
- `state_hash` changes across updates.
